# FGVC-Aircraft: ResNet-50 weights V1 against V2

BreastMNIST uses torchvision's `IMAGENET1K_V2` weights, the default
(`../../medmnist_breastmnist/demos/weights.ipynb`). On FGVC-Aircraft this
project uses the older V1. This notebook shows why, and checks that the
choice does not depend on the test set.

The protocol is that of `configs/aircraft70_gpu.yaml`:

- every non-constant feature, standardized on the training set;
- logistic regression, `max_iter=1000`, C = 1;
- 16 random splits, 70% of the 1,198 photos in the training set, seed 42
  (two near-identical shots dropped, `duplicates.ipynb`).

The choice was first made by test AUC (`scripts/aircraft_weights.py`). Those
16 test sets also test the selection methods, so the choice could have
used the test set. The second part makes the same choice on validation sets
inside each training set, as the runner chooses `k`
(`scripts/aircraft_weights_validation.py`). The code of both parts is
imported from those two scripts.

Both embedding files are cached beside the loader. Without them, the first
call reads the 2.75 GB archive (minutes on a GPU).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT))              # repo root, so featsel_datasets imports
sys.path.insert(0, str(ROOT / 'scripts'))  # the two scripts this notebook reuses

from aircraft_weights import CONFIG, WEIGHTS
from aircraft_weights_validation import N_SPLITS, SEED, TRAIN_SIZE, validation_auc

from featsel.run import run
from featsel_datasets.fgvc_aircraft.aircraft import FGVCAircraft
from featsel_datasets.fgvc_aircraft.featsel_view import NAME, TARGET, load_xy

data = {w: FGVCAircraft(weights=w) for w in WEIGHTS}
xy = {w: load_xy(d, drop_copies=True) for w, d in data.items()}

## The features

The share of exact zeros among all values of the 1,198 photos, and the
number of features left after the constant ones are dropped.

In [2]:
pd.DataFrame({w: {'zero share': (data[w].embeddings.loc[y.index].to_numpy() == 0).mean(),
                  'non-constant features': X.shape[1]}
              for w, (X, y) in xy.items()}).T

,zero share,non-constant features
V1,0.001436,2048.0
V2,0.802641,2048.0


## Test AUC with all features

The original comparison. Random selection of every feature is the
runner's all-features reference. Each run's CSV goes to `results/local/`.

In [3]:
test = {}
for w, (X, y) in xy.items():
    config = {**CONFIG,
              'selectors': [{'name': 'RandomSelector', 'k': [X.shape[1]]}],
              'output': str(ROOT / 'results' / 'local' / f'aircraft_weights_{w.lower()}.csv')}
    test[w] = pd.read_csv(run(X, y, config, target=TARGET, name=f'{NAME} {w}'))['roc_auc']
test = pd.DataFrame(test)
test.describe().loc[['mean', '25%', '50%', '75%']]


=== /home/dror/projects/featsel/results/local/aircraft_weights_v1.csv ===
Dataset      : FGVC-Aircraft ResNet-50 V1, 1200 samples x 2048 features, target 'family'
Seed         : 42
Tasks        : 1
  binary                 Boeing 737 800, A320 400
Splits       : 16 stratified Monte Carlo draws
  train_size 0.7: 840 train / 360 test
Repeats      : 1 fits per selection, median time recorded
Preprocess   : standard
Metrics      : roc_auc
Workers      : 1 (parallel over task x selector jobs, one shared copy of each split)
Memory cap   : 8.9 GB of data per worker
Selectors    :
  RandomSelector   k = 2048 (100.00%)
Models       :
  logistic_regression {'max_iter': 1000}

Total        : 16 executions (1 tasks x 16 splits x 1 train sizes x 1 preprocess x 1 selector-k x 1 models)



  0%|          | 0/16 [00:00<?, ?exec/s]

Wrote 16 executions to /home/dror/projects/featsel/results/local/aircraft_weights_v1.csv in 24s

=== /home/dror/projects/featsel/results/local/aircraft_weights_v2.csv ===
Dataset      : FGVC-Aircraft ResNet-50 V2, 1200 samples x 2048 features, target 'family'
Seed         : 42
Tasks        : 1
  binary                 Boeing 737 800, A320 400
Splits       : 16 stratified Monte Carlo draws
  train_size 0.7: 840 train / 360 test
Repeats      : 1 fits per selection, median time recorded
Preprocess   : standard
Metrics      : roc_auc
Workers      : 1 (parallel over task x selector jobs, one shared copy of each split)
Memory cap   : 8.9 GB of data per worker
Selectors    :
  RandomSelector   k = 2048 (100.00%)
Models       :
  logistic_regression {'max_iter': 1000}

Total        : 16 executions (1 tasks x 16 splits x 1 train sizes x 1 preprocess x 1 selector-k x 1 models)



  0%|          | 0/16 [00:00<?, ?exec/s]

Wrote 16 executions to /home/dror/projects/featsel/results/local/aircraft_weights_v2.csv in 15s


,V1,V2
mean,0.862140,0.741986
25%,0.850278,0.724670
50%,0.870469,0.739792
75%,0.877135,0.754453


## The same choice on validation sets

On each split, the training set alone is cut into 5 stratified folds. Each
version gets its mean validation AUC; the higher one wins the split. The
test set is not used.

In [4]:
y = pd.factorize(xy['V1'][1], sort=True)[0]
arrays = {w: np.asarray(X, dtype=np.float32) for w, (X, _) in xy.items()}
splitter = StratifiedShuffleSplit(n_splits=N_SPLITS, train_size=TRAIN_SIZE, random_state=SEED)
validation = pd.DataFrame(
    [{w: validation_auc(arrays[w], y, train_idx, SEED + split) for w in WEIGHTS}
     for split, (train_idx, _) in enumerate(splitter.split(arrays['V1'], y))])
validation.describe().loc[['mean', 'min', 'max']]

,V1,V2
mean,0.851678,0.753704
min,0.823501,0.712309
max,0.878922,0.785459


## Both choices, split by split

In [5]:
pd.DataFrame({
    'test AUC, V1 wins': [(test['V1'] > test['V2']).sum()],
    'validation AUC, V1 wins': [(validation['V1'] > validation['V2']).sum()],
    'splits': [N_SPLITS],
    'test AUC, V1 - V2': [(test['V1'] - test['V2']).mean()],
    'validation AUC, V1 - V2': [(validation['V1'] - validation['V2']).mean()],
}, index=['FGVC-Aircraft']).round(3)

,"test AUC, V1 wins","validation AUC, V1 wins",splits,"test AUC, V1 - V2","validation AUC, V1 - V2"
FGVC-Aircraft,16,16,16,0.12,0.098


## Conclusion

On FGVC-Aircraft, V1 is the right choice, and the choice does not depend
on the test set.

- **Features:** 80% of V2's values are exact zeros, against 0.1% of V1's.
- **Test AUC:** 0.862 with V1 against 0.742 with V2. V1 wins all 16
  splits.
- **Validation AUC**, inside each training set only: 0.852 against 0.754.
  V1 wins all 16 splits again, so the same choice is made without the
  test set.

On BreastMNIST the better version is V2
(`../../medmnist_breastmnist/demos/weights.ipynb`). The better weights
depend on the dataset, so each new dataset needs this check, made on
validation sets.